# 5 · Off-design performance and the real engines

The calibrated CFM56-7B26/3 and LEAP-1B28 models (ICAO databank data, one calibration each),
their LTO fuel flows, a cruise point, a hot day and degradation signatures.
Lesson: [05](../docs/lessons/05-off-design.md) ·
Theory: [off-design](../docs/off_design_performance.md) ·
[flight parameters](../docs/flight-parameters.md).

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from propulsion import plotting as P
from propulsion.flight import FlightCondition
from propulsion.units import FT, tsfc_to_lb_per_lbf_h

P.use_style()
np.set_printoptions(precision=4, suppress=True)
FL350 = FlightCondition.at_altitude(35000 * FT, 0.78, geopotential=True)
SLS = FlightCondition.sea_level_static()
sys.path.insert(0, "../validation")  # calibrated engine models and cited data
from engines import calibrated_engine, ENGINES
from reference import icao, tcds, LTO_MODES

## Calibrated design points

In [2]:
for name in ENGINES:
    d = calibrated_engine(name).design_point
    print(f"{name:13s} BPR {d.alpha:.2f}  OPR {d.pi_c:.2f}  fan PR {d.pi_f:.3f}  "
          f"Tt4 {d.Tt4:.1f} K  airflow {d.mdot0:.1f} kg/s  TSFC {d.tsfc*1e6:.3f} mg/(N s)")

CFM56-7B26/3  BPR 5.10  OPR 27.70  fan PR 1.808  Tt4 1417.3 K  airflow 348.6 kg/s  TSFC 10.368 mg/(N s)
LEAP-1B28     BPR 8.24  OPR 42.03  fan PR 1.523  Tt4 1492.3 K  airflow 475.2 kg/s  TSFC 8.205 mg/(N s)


## ICAO LTO fuel flow: take-off is calibrated, the rest are predictions

In [3]:
for name in ENGINES:
    e, d = calibrated_engine(name), icao(name)
    for mode, frac in LTO_MODES:
        p = e.design_point if frac == 1.0 else e.operate_at_thrust(SLS, frac * d["rated_thrust"])
        ref = d["ff_" + mode]
        print(f"{name:13s} {mode:9s} model {p.fuel_flow:.4f}  ICAO {ref:.4f} kg/s  "
              f"({100*(p.fuel_flow/ref-1):+5.1f} %)  N1 {100*p.N1:5.1f} %  N2 {100*p.N2:5.1f} %")

CFM56-7B26/3  takeoff   model 1.2130  ICAO 1.2130 kg/s  ( +0.0 %)  N1 100.0 %  N2 100.0 %
CFM56-7B26/3  climbout  model 0.9965  ICAO 0.9860 kg/s  ( +1.1 %)  N1  93.4 %  N2  97.1 %
CFM56-7B26/3  approach  model 0.3111  ICAO 0.3310 kg/s  ( -6.0 %)  N1  58.7 %  N2  80.9 %
CFM56-7B26/3  idle      model 0.0876  ICAO 0.1080 kg/s  (-18.8 %)  N1  31.0 %  N2  66.2 %
LEAP-1B28     takeoff   model 1.0700  ICAO 1.0700 kg/s  ( +0.0 %)  N1 100.0 %  N2 100.0 %
LEAP-1B28     climbout  model 0.8777  ICAO 0.8700 kg/s  ( +0.9 %)  N1  93.1 %  N2  97.1 %
LEAP-1B28     approach  model 0.2719  ICAO 0.2897 kg/s  ( -6.1 %)  N1  58.4 %  N2  81.2 %
LEAP-1B28     idle      model 0.0792  ICAO 0.1004 kg/s  (-21.1 %)  N1  32.2 %  N2  67.4 %


## Cruise: corrected parameters

In [4]:
e = calibrated_engine("CFM56-7B26/3")
p = e.operate_at_thrust(FL350, 22e3)
print(f"Tt4 = {p.Tt4:.0f} K (Tt4/theta0 = {p.Tt4/(FL350.T0/288.15):.0f} K), alpha = {p.alpha:.2f}, "
      f"OPR = {p.pi_c:.1f}")
print(f"N1 = {100*p.N1:.1f} %, corrected N1 = {100*p.N1_corrected:.1f} %, "
      f"corrected flow = {p.corrected_flow:.1f} kg/s (take-off: {e.design_point.corrected_flow:.1f})")
print(f"TSFC = {p.tsfc*1e6:.2f} mg/(N s) = {tsfc_to_lb_per_lbf_h(p.tsfc):.3f} lb/(lbf h)")

Tt4 = 1195 K (Tt4/theta0 = 1574 K), alpha = 5.18, OPR = 27.2
N1 = 92.8 %, corrected N1 = 100.5 %, corrected flow = 352.0 kg/s (take-off: 350.4)
TSFC = 17.12 mg/(N s) = 0.604 lb/(lbf h)


## Hot day and component wear

In [5]:
hot = FlightCondition.sea_level_static(dT=15.0)
same = e.operate(hot, e.design_point.Tt4)
rated = e.operate_at_thrust(hot, e.design_point.thrust, start=same)
print(f"ISA+15 at the take-off Tt4: {100*same.thrust/e.design_point.thrust:.1f} % of rated thrust")
print(f"rated thrust needs Tt4 = {rated.Tt4:.0f} K and raises Tt45 by {rated.Tt45 - e.design_point.Tt45:.0f} K")
base = e.operate_at_thrust(SLS, e.design_point.thrust)
for field in ("e_f", "e_c", "e_t"):
    worn = e.with_components(e.components.with_(**{field: getattr(e.components, field) - 0.01}))
    q = worn.operate_at_thrust(SLS, e.design_point.thrust)
    print(f"{field} - 0.01: dTt45 = {q.Tt45 - base.Tt45:+5.1f} K, dWf = {100*(q.fuel_flow/base.fuel_flow-1):+.2f} %, "
          f"dN1 = {100*(q.N1-base.N1):+.2f}, dN2 = {100*(q.N2-base.N2):+.2f} % pts")

ISA+15 at the take-off Tt4: 86.8 % of rated thrust
rated thrust needs Tt4 = 1490 K and raises Tt45 by 56 K
e_f - 0.01: dTt45 =  +5.0 K, dWf = +0.92 %, dN1 = +0.49, dN2 = +0.24 % pts
e_c - 0.01: dTt45 = +15.0 K, dWf = +0.78 %, dN1 = +0.12, dN2 = +0.71 % pts
e_t - 0.01: dTt45 = +14.1 K, dWf = +1.35 %, dN1 = -0.05, dN2 = -0.03 % pts
